<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/wip-cartpole-dqn-lightning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

- TODO: compare dueling dqn vs dqn
- TODO: Replace the two nn.Linear layers in each stream with NoisyLinear; drop ϵ-greedy once training kicks off.
- TODO: Store n-length trajectories in the replay (Tianshou has NStepCollector & NStepPrioritizedReplayBuffer).
- TODO: Swap the single-value head for a categorical head (51 atoms is the canonical choice).
- TODO: make sure to record if training is stopped early (eg: target reached)
- TODO: Checkpoint the best model via ModelCheckpoint(monitor="reward_mean", save_top_k=1).
- TODO: Add gradient_clip_val=10.0 and precision="16-mixed" to pl.Trainer.

# Solve Gymnasium with Rainbow

This notebook trains a **Deep Q-Network (DQN)** agent on the classic environment using **PyTorch Lightning**.

Install required libraries for Gymnasium, PyTorch Lightning, Tianshou, WandB, and notebook utilities.

In [1]:
%pip install gymnasium[classic-control] pytorch-lightning tianshou wandb[media]>=0.20 tsilva_notebook_utils==0.0.99 > /dev/null

Note: you may need to restart the kernel to use updated packages.


Load API keys and authentication tokens from Colab secrets for secure access.

🔑 Loading API keys and authentication tokens from Colab secrets:

In [2]:
from tsilva_notebook_utils.colab import load_secrets_into_env

_ = load_secrets_into_env([
    'WANDB_API_KEY',
    'NOTIFICATION_URL',
    'NOTIFICATION_AUTH_TOKEN'
])

Define the configuration for the environment and agent hyperparameters.

In [3]:
import torch.nn as nn
from typing import Dict

def setup_config(env_id: str = "CartPole-v1") -> Dict[str, object]:
    # ------------------------------------------------------------------ #
    # Defaults that are reasonable for *most* small, discrete-state tasks
    # ------------------------------------------------------------------ #
    common: Dict[str, object] = dict(
        env_id=env_id,
        seed=42,
        max_epochs=-1,                 # stop via reward threshold instead
        max_steps=-1,
        log_every_n_steps=10,                   # log every 10 env-steps
        normalize_obs=False,
        buffer_size=10_000,
        per_alpha=0.0,              # 0 → vanilla replay buffer
        per_beta_start=0.4,
        per_beta_end=1.0,
        eval_every_n_episodes=10,
        running_reward_window=100,             # running-mean window
        target_update_interval=200,        # hard-update interval
        target_soft_tau=0.005,                     # soft-update coefficient
        gradient_clip_val=None,        # e.g. 0.5 to enable clipping
    )

    # ------------------------------------------------------------------ #
    # Environment-specific overrides
    # ------------------------------------------------------------------ #
    env_specific: Dict[str, Dict[str, object]] = {
        "CartPole-v1": dict(
            hidden_dims=(32,),
            discount_factor=0.99,
            batch_size=32,
            min_replay_size=500,
            buffer_size=2_000,
            learning_rate=5e-4,
            epsilon_start=1.0,
            epsilon_end=0.05,
            epsilon_decay_steps=2_000,
            target_update_interval=100,
            target_soft_tau=0.0,
            gradient_clip_val=5.0,
            reward_threshold=475.0
        ),
        "MountainCar-v0": dict(
            hidden_dims=(256, 256),
            discount_factor=0.99,
            batch_size=64,
            min_replay_size=5_000,
            buffer_size=100_000,
            learning_rate=2.5e-4,
            epsilon_start=1.0,
            epsilon_end=0.01,
            epsilon_decay_steps=50_000,
            reward_threshold=-110.0
        ),
    }

    if env_id not in env_specific:
        raise ValueError(f"Unsupported env_id: {env_id}")

    # Merge: environment block overrides any duplicate keys in *common*
    return {**common, **env_specific[env_id]}

CONFIG = setup_config()

In [4]:
# --- Runtime metadata --------------------------------------------------------
import subprocess, torch, platform, os

def _get_git_commit() -> str:
    """Return the short SHA if this is a Git repo, else 'unknown'."""
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception:          # not a Git checkout or Git not installed
        return "unknown"

def runtime_metadata():
    return {
        "git_commit": _get_git_commit(),
        "torch_version": torch.__version__,
        "torch_cuda": torch.version.cuda or "cpu",
        "cuda_device": (
            torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
        ),
        "python_version": platform.python_version(),
        "run_host": os.uname().nodename,
    }

# Merge into the notebook-wide CONFIG dict
CONFIG.update(runtime_metadata())

In [5]:
CONFIG

{'env_id': 'CartPole-v1',
 'seed': 42,
 'max_epochs': -1,
 'max_steps': -1,
 'log_every_n_steps': 10,
 'normalize_obs': False,
 'buffer_size': 2000,
 'per_alpha': 0.0,
 'per_beta_start': 0.4,
 'per_beta_end': 1.0,
 'eval_every_n_episodes': 10,
 'running_reward_window': 100,
 'target_update_interval': 100,
 'target_soft_tau': 0.0,
 'gradient_clip_val': 5.0,
 'hidden_dims': (32,),
 'discount_factor': 0.99,
 'batch_size': 32,
 'min_replay_size': 500,
 'learning_rate': 0.0005,
 'epsilon_start': 1.0,
 'epsilon_end': 0.05,
 'epsilon_decay_steps': 2000,
 'reward_threshold': 475.0,
 'git_commit': '88b4bdb',
 'torch_version': '2.5.1',
 'torch_cuda': '11.8',
 'cuda_device': 'NVIDIA GeForce RTX 2060',
 'python_version': '3.11.11',
 'run_host': 'BEAST-2'}

Set the random seed for reproducibility.

In [6]:
from tsilva_notebook_utils.lightning import seed_everything
seed_everything(CONFIG['seed'])

Seed set to 42


42

Login to Weights & Biases (wandb) for experiment tracking.

In [7]:
from wandb import login
login()

wandb: Currently logged in as: tsilva to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

Build and test the Gymnasium environment using the provided configuration.

In [8]:
from tsilva_notebook_utils.misc import filter_kwargs
from tsilva_notebook_utils.gymnasium import build_env

env, _, _ = build_env(**filter_kwargs(build_env, CONFIG))
env

<TimeLimit<OrderEnforcing<PassiveEnvChecker<CartPoleEnv<CartPole-v1>>>>>

Initialize the environment and determine the input and output dimensions for the model.

In [9]:
env, _, _ = build_env(**filter_kwargs(build_env, CONFIG))
N_INPUTS = env.observation_space.shape[0]
N_OUTPUTS = int(env.action_space.n)
N_INPUTS, N_OUTPUTS

(4, 2)

Create DQN model:

In [10]:
class DQNModel(nn.Module):
    def __init__(self, n_inputs, hidden_dims, n_outputs):
        super().__init__()

        layers = []
        input_size = n_inputs
        for hidden_size in hidden_dims:
            layers.append(nn.Linear(input_size, hidden_size))
            layers.append(nn.ReLU())
            input_size = hidden_size
        layers.append(nn.Linear(input_size, n_outputs))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)
    
model = DQNModel(N_INPUTS, CONFIG['hidden_dims'], N_OUTPUTS)
model

DQNModel(
  (net): Sequential(
    (0): Linear(in_features=4, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=2, bias=True)
  )
)

Create Dueling DQN model:

In [11]:
import torch.nn as nn
import torch

class DuelingDQNModel(nn.Module):
    """
    A feed-forward dueling network:
        • shared feature extractor
        • separate value (V) and advantage (A) streams
        • Q(s,a) = V(s) + A(s,a) − mean_a A(s,a)
    """
    def __init__(self, n_inputs, hidden_dims, n_outputs):
        super().__init__()

        # --- shared feature layers ----------------------------------------
        layers = []
        last = n_inputs
        for h in hidden_dims:
            layers += [nn.Linear(last, h), nn.ReLU()]
            last = h
        self.feature = nn.Sequential(*layers)

        # --- value stream --------------------------------------------------
        self.value = nn.Sequential(
            nn.Linear(last, last),
            nn.ReLU(),
            nn.Linear(last, 1),
        )

        # --- advantage stream ---------------------------------------------
        self.advantage = nn.Sequential(
            nn.Linear(last, last),
            nn.ReLU(),
            nn.Linear(last, n_outputs),
        )

    # ---------------------------------------------------------------------
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not torch.is_tensor(x):                       # allow NumPy inputs
            x = torch.as_tensor(x, dtype=torch.float32)
        f = self.feature(x)
        v = self.value(f)                      # shape: (B, 1)
        a = self.advantage(f)                  # shape: (B, A)
        q = v + a - a.mean(dim=1, keepdim=True)
        return q

model = DuelingDQNModel(N_INPUTS, CONFIG['hidden_dims'], N_OUTPUTS)
model

DuelingDQNModel(
  (feature): Sequential(
    (0): Linear(in_features=4, out_features=32, bias=True)
    (1): ReLU()
  )
  (value): Sequential(
    (0): Linear(in_features=32, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=1, bias=True)
  )
  (advantage): Sequential(
    (0): Linear(in_features=32, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=2, bias=True)
  )
)

Implement the PyTorch Lightning module for DQN training, including replay buffer and training logic.

In [12]:
import random
import numpy as np
import pytorch_lightning as pl
from tianshou.data import Batch, ReplayBuffer, PrioritizedReplayBuffer
from tsilva_notebook_utils.torch import create_infinite_data_loader

infinite_data_loader = create_infinite_data_loader()

class DQNModule(pl.LightningModule):
    def __init__(self):
        super().__init__()

        self.save_hyperparameters()

        self.q_model = torch.compile(DQNModel(N_INPUTS, CONFIG['hidden_dims'], N_OUTPUTS).to(self.device), mode="default", fullgraph=True)
        self.target_model = torch.compile(DQNModel(N_INPUTS, CONFIG['hidden_dims'], N_OUTPUTS).to(self.device), mode="default", fullgraph=True)
        self.target_model.load_state_dict(self.q_model.state_dict())

        self.build_env_fn = lambda **kwargs: build_env(**filter_kwargs(build_env, {**CONFIG, **kwargs}))
        self.env, self.state, _ = self.build_env_fn()
        
        per_alpha = CONFIG['per_alpha']
        if per_alpha > 0: self.buffer = PrioritizedReplayBuffer(size=CONFIG['buffer_size'], alpha=CONFIG['per_alpha'], beta=CONFIG['replay_beta'])
        else: self.buffer = ReplayBuffer(size=CONFIG["buffer_size"])
        
        self.episode = 0
        self.total_steps = 0
        self.episode_steps = 0
        self.episode_reward = 0
        self.episode_shaped_reward = 0
        self.episode_rewards = []

    def forward(self, x):
        return self.q_model(x)

    @property
    def current_eps(self):
        epsilon_end = CONFIG['epsilon_end']
        epsilon_start = CONFIG['epsilon_start']
        epsilon_decay_steps = CONFIG['epsilon_decay_steps']
        eps = max(epsilon_end, epsilon_start - (epsilon_start - epsilon_end) * (self.total_steps / epsilon_decay_steps))
        return eps
    
    def act(self, state):
        if random.random() < self.current_eps:
            return self.env.action_space.sample()
        else:
            state = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
            with torch.no_grad(): q = self.q_model(state)
            return int(torch.argmax(q, dim=1)[0].item())

    def train_dataloader(self):
        return infinite_data_loader

    @torch.no_grad()
    def soft_update(self, target_soft_tau):
        for target_param, online_param in zip(self.target_model.parameters(), self.q_model.parameters()):
            target_param.data.copy_(target_soft_tau * online_param.data + (1.0 - target_soft_tau) * target_param.data)
    
    @torch.no_grad()
    def hard_update(self):
        self.target_model.load_state_dict(self.q_model.state_dict())

    @property
    def current_beta(self) -> float:
        frac = min(1.0, self.total_steps / CONFIG['epsilon_decay_steps'])
        return CONFIG['per_beta_start'] + frac * (
            CONFIG['per_beta_end'] - CONFIG['per_beta_start']
        )

    def training_step(self, batch, batch_idx):
        action = self.act(self.state)
        next_state, reward, terminated, truncated, info = self.env.step(action)
        shaped_reward = reward

        done = terminated or truncated
        self.buffer.add(Batch(
            obs=self.state,
            act=action,
            rew=shaped_reward,
            terminated=terminated,
            truncated=truncated,
            done=done,
            obs_next=next_state,
            info=info
        ))

        self.state = next_state

        self.total_steps += 1
        self.episode_steps += 1
        self.episode_reward += reward
        self.episode_shaped_reward += shaped_reward

        per_alpha = CONFIG['per_alpha']
        use_per = per_alpha > 0

        loss = None
        if len(self.buffer) >= CONFIG['min_replay_size']:
            if use_per: 
                self.buffer.set_beta(self.current_beta)
                batch, indices = self.buffer.sample(CONFIG['batch_size'])
            else: 
                batch, _ = self.buffer.sample(CONFIG["batch_size"])

            states = torch.tensor(batch.obs, dtype=torch.float32, device=self.device)
            actions = torch.tensor(batch.act, dtype=torch.long, device=self.device).unsqueeze(-1)
            rewards = torch.tensor(batch.rew, dtype=torch.float32, device=self.device)
            next_states = torch.tensor(batch.obs_next, dtype=torch.float32, device=self.device)
            dones = torch.tensor(batch.done, dtype=torch.float32, device=self.device)
            if use_per: weights = torch.tensor(batch.weight, dtype=torch.float32, device=self.device)
            else: weights = torch.ones_like(rewards, device=self.device)
            
            q_values = self.q_model(states).gather(1, actions).squeeze()

            # Double-DQN target: online net chooses the action, target net evaluates it
            next_online_actions = self.q_model(next_states).argmax(1, keepdim=True)
            next_q = self.target_model(next_states).gather(1, next_online_actions).squeeze()

            targets = rewards + CONFIG['discount_factor'] * next_q * (1 - dones)

            if use_per: 
                td_errors = (q_values - targets.detach()).abs()
                priorities = (td_errors + 1e-6) ** CONFIG["per_alpha"]
                self.buffer.update_weight(indices, priorities.cpu().numpy())

            # Optionally use importance-sampling weights for loss
            #loss = (weights * nn.functional.mse_loss(q_values, targets.detach(), reduction='none')).mean()
            loss = (weights * nn.functional.smooth_l1_loss(q_values, targets.detach(), reduction='none')).mean()
            self.log('loss', loss, on_step=True, prog_bar=True)
            
            target_soft_tau = CONFIG['target_soft_tau']
            target_update_interval = CONFIG['target_update_interval']
            use_hard_update = target_update_interval > 0
            use_soft_update = not use_hard_update and target_soft_tau > 0
            if use_hard_update and self.total_steps % target_update_interval == 0: self.hard_update()
            elif use_soft_update: self.soft_update(target_soft_tau)
                
        if done:
            self.episode_rewards.append(self.episode_reward)
            
            # Compute stats
            rewards_arr = np.array(self.episode_rewards[-CONFIG['running_reward_window']:])
            min_r = float(np.min(rewards_arr))
            max_r = float(np.max(rewards_arr))
            mean_r = float(np.mean(rewards_arr))
            std_r = float(np.std(rewards_arr))

            # Log stats
            self.log('episode', self.episode, on_step=True, prog_bar=True)
            self.log('reward', self.episode_reward, on_step=True, prog_bar=True)
            self.log('shaped_reward', self.episode_shaped_reward, on_step=True, prog_bar=True)
            self.log('steps', self.episode_steps, on_step=True, prog_bar=True)
            self.log('eps', self.current_eps, on_step=True, prog_bar=True)
            self.log('reward_min', min_r, on_step=True, prog_bar=True)
            self.log('reward_max', max_r, on_step=True, prog_bar=True)
            self.log('reward_mean', mean_r, on_step=True, prog_bar=True)
            self.log('reward_std', std_r, on_step=True, prog_bar=True)
            if use_per: self.log('beta', self.current_beta, on_step=True, prog_bar=True)

            self.episode_steps = 0
            self.episode_reward = 0
            self.episode_shaped_reward = 0
            self.episode += 1
            self.state = self.env.reset()[0]

        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.q_model.parameters(), lr=CONFIG["learning_rate"])

    """
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.q_model.parameters(), learning_rate=CONFIG["learning_rate"])

        # First cosine cycle lasts ≈ one fifth of the intended training horizon.
        # Feel free to tweak T_0/T_mult if you plan huge runs.
        scheduler = torch.optim.learning_rate_scheduler.CosineAnnealingWarmRestarts(
            optimizer,
            T_0=10_000,           # 10 k env-steps for the first cycle
            T_mult=2,             # double the cycle length at every restart
            eta_min=CONFIG["learning_rate"] * 0.1  # floor at 10 % of the initial learning_rate
        )

        # Lightning expects a dict when the scheduler isn’t epoch-based.
        return {
            "optimizer": optimizer,
            "learning_rate_scheduler": {
                "scheduler": scheduler,
                "interval": "step",   # call scheduler.step() every training_step
                "frequency": 1,
            },
        }
    """

module = DQNModule()
module

DQNModule(
  (q_model): OptimizedModule(
    (_orig_mod): DQNModel(
      (net): Sequential(
        (0): Linear(in_features=4, out_features=32, bias=True)
        (1): ReLU()
        (2): Linear(in_features=32, out_features=2, bias=True)
      )
    )
  )
  (target_model): OptimizedModule(
    (_orig_mod): DQNModel(
      (net): Sequential(
        (0): Linear(in_features=4, out_features=32, bias=True)
        (1): ReLU()
        (2): Linear(in_features=32, out_features=2, bias=True)
      )
    )
  )
)

Set up the PyTorch Lightning trainer and start training the DQN agent.

In [ ]:
import os
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger

from tsilva_notebook_utils.lightning import StopOnLambda
from tsilva_notebook_utils.gymnasium import build_pl_callback

trainer = pl.Trainer(
    max_epochs=CONFIG['max_epochs'],
    max_steps=CONFIG['max_steps'],
    log_every_n_steps=CONFIG['log_every_n_steps'],
    logger=WandbLogger(project=os.getenv('NOTEBOOK_ID'), config=CONFIG),
    enable_model_summary=False,
    gradient_clip_val=CONFIG['gradient_clip_val'],
    callbacks=[
        StopOnLambda(
            lambda metrics: metrics.get('reward_mean', -float('inf')) >= CONFIG['reward_threshold'],
            message=f"Stopping: reward_mean >= {CONFIG['reward_threshold']}"
        ),
        build_pl_callback("EvalEpisodeAndRecordCallback", every_n_episodes=CONFIG['eval_every_n_episodes'])
    ]
)
trainer.fit(module)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/tsilva/miniconda3/envs/aiml-notebooks/lib/python3.11/site-packages/wandb/analytics/sentry.py:258: DeprecationWarning: The `Scope.user` setter is deprecated in favor of `Scope.set_user()`.
  self.scope.user = {"email": email}
/home/tsilva/miniconda3/envs/aiml-notebooks/lib/python3.11/site-packages/wandb/analytics/sentry.py:258: DeprecationWarning: The `Scope.user` setter is deprecated in favor of `Scope.set_user()`.
  self.scope.user = {"email": email}


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Training: |          | 0/? [00:00<?, ?it/s]

/home/tsilva/miniconda3/envs/aiml-notebooks/lib/python3.11/site-packages/pytorch_lightning/loops/optimization/automatic.py:134: `training_step` returned `None`. If this was on purpose, ignore this warning...
/home/tsilva/miniconda3/envs/aiml-notebooks/lib/python3.11/site-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


Render and visualize a trained episode using the learned Q-network.

In [ ]:
from tsilva_notebook_utils.gymnasium import render_episode

render_episode(
    env=module.build_env_fn,
    model=module.q_model
)